In [1]:
import numpy as np
import os
import rasterio as rio
from rasterio.features import rasterize
from rasterio.enums import Resampling
from scipy.ndimage import zoom
import pickle as pkl

#from c2s.models.chicken import ChickenConfig, ChickenOutput, Chicken

In [2]:
site = 'CS-103A'
run_name = f'{site}_01'

# location of raw lichen data
data_dir = '/network/scratch/m/matthew.fortier/shared/lichen/raw'

# place to store temporary files
run_dir = f'/network/scratch/m/matthew.fortier/runs/{run_name}'

In [3]:
if not os.path.exists(run_dir):
    os.makedirs(os.path.join(run_dir, 'composite')

In [4]:
def load_site_data(site: str):
    # Retrieve class, RGB, canopy height, and certainty data from source files
    # Output them as a 6 layer raster (class, R, G, B, chm, certainty)
    
    rgb_file = os.path.join(data_dir, site, f'{site}_hp_transparent_mosaic_group1.tif')
    chm_file = os.path.join(data_dir, site, f'{site}_hp_chm_stratified.tif')
    
    rasters = {}

    # load RGB rasters and metadata
    with rio.open(rgb_file) as f:
        rasters['meta'] = f.meta
        rasters['R'] = f.read((1)).astype(float)/255.0
        rasters['G'] = f.read((2)).astype(float)/255.0
        rasters['B'] = f.read((3)).astype(float)/255.0
        
    # load chm, resize to match RGB rasters
    with rio.open(chm_file) as f:
        height_scale = rasters['meta']['height'] / f.height
        width_scale = rasters['meta']['width'] / f.width
        rasters['chm'] = zoom(f.read((1)).astype(np.int8), (height_scale, width_scale), order=0)

    return rasters

In [5]:
rasters = load_site_data('CS-103A')

In [6]:
eps = 1e-9
chunks = 4 # cuts into 4x4 chunks. keep this as small as possible
chunk_height = rasters['meta']['height'] / chunks
chunk_width = rasters['meta']['width'] / chunks

# Features - to save memory, I'm allocating the array in advance
# and using pseudo enumerators to index the layers
R = 0
G = 1
B = 2
chm = 3
rc = 4
gc = 5
bc = 6
rc_d_gc = 7
rc_p_gc = 8
ExG = 9
ExR = 10
ExB = 11
ExGmExR = 12
Ikaw = 13
MGRVI = 14
GLI = 15
Y = 16
L = 17
z_score_Y = 18
z_score_L = 19

for y in range(chunks):
    for x in range(chunks):
        y_min = int(y * chunk_height)
        y_max = int(min((y+1) * chunk_height, rasters['meta']['height']))

        x_min = int(x * chunk_width)
        x_max = int(min((x+1) * chunk_width, rasters['meta']['width']))
        
        # 20 features total
        chunk = np.zeros((20, y_max-y_min, x_max-x_min))
        
        chunk[R] = rasters['R'][y_min:y_max,x_min:x_max]
        chunk[G] = rasters['G'][y_min:y_max,x_min:x_max]
        chunk[B] = rasters['B'][y_min:y_max,x_min:x_max]
        chunk[chm] = rasters['chm'][y_min:y_max,x_min:x_max]
        
        # shortcircuit if there are no valid pixels
        check = np.where(chunk[chm] >= 0)
        if len(check[0]) == 0:
            print(f'Skipping chunk {y}-{x}')
            continue
   
        chunk[rc] = chunk[R] / (chunk[R] + chunk[G] + chunk[B] + eps)
        chunk[gc] = chunk[G] / (chunk[R] + chunk[G] + chunk[B] + eps)


        # --------------------------------------------------------------------------------------
        # extending pada data frame by rc/gc, rc+gc, luminance Y, perceived lightness L, z_score_Y, z_score_L 
        # Excess indecies ExG, ExR, ExB, indices ExGmExR, Ikaw, MGRVI, and GLI 
        # --------------------------------------------------------------------------------------
        # blue chromaticity
        chunk[bc] = 1-(chunk[rc]+chunk[gc])

        # ration rc/gc
        chunk[rc_d_gc] = chunk[rc]/(chunk[gc] + eps)

        # sum rc+gc
        chunk[rc_p_gc] = chunk[rc]+chunk[gc]

        # excess indecies
        chunk[ExG] = 2*chunk[G] - chunk[R] - chunk[B]

        chunk[ExR] = 1.4*chunk[R] - chunk[G]

        chunk[ExB] = 1.4*chunk[B] - chunk[G]

        # index ExGmExR
        chunk[ExGmExR] = chunk[ExG]-chunk[ExR]

        # index Ikaw - Kawashima Index
        chunk[Ikaw] = (chunk[R]-chunk[B])/(chunk[R]+chunk[B]+eps)

        # MGRVI - modified green red vegetation index
        chunk[MGRVI] = ((chunk[G])**2 - (chunk[R])**2)/((chunk[G])**2 + (chunk[R])**2 + eps)

        # GLI - green leaf index
        chunk[GLI] = (2*chunk[G] - chunk[R] - chunk[B])/(2*chunk[G] + chunk[R] + chunk[B] + eps)

        # luminance Y
        sRGBtoLin = lambda x: np.where(x <= 0.04045, x/12.92, (pow(((x + 0.055)/1.055), 2.4)))
        chunk[Y] = 0.2126*sRGBtoLin(chunk[R]) + 0.7152*sRGBtoLin(chunk[G]) + 0.0722*sRGBtoLin(chunk[B])

        # perceived lightness L
        YtoLstar = lambda y: np.where(y <= (216/24389), y*(24389/27), (pow(y, (1/3)) * 116 - 16))
        chunk[L] = YtoLstar(chunk[Y])

        # z_scores for Y and L
        valid_Y_np = chunk[Y][check]
        Y_mean = valid_Y_np.mean()
        Y_std = valid_Y_np.std()
        del valid_Y_np

        valid_L_np = chunk[L][check]
        L_mean = valid_L_np.mean()
        L_std = valid_L_np.std()
        del valid_L_np

        chunk[z_score_Y] = (chunk[Y] - Y_mean)/(Y_std+eps)
        chunk[z_score_L] = (chunk[L] - L_mean)/(L_std+eps)
        file_path = os.path.join(run_dir, 'composite' f'chunk_{y}_{x}.pkl')
        print(f'Writing chunk {y}-{x}')
        with open(file_path, "wb") as file:
            # Use the pickle.dump() function to write the dictionary to the file
            pkl.dump(chunk, file)

Skipping chunk 0-0
Writing chunk 0-1
Writing chunk 0-2
Writing chunk 0-3
Writing chunk 1-0
Writing chunk 1-1
Writing chunk 1-2
Writing chunk 1-3
Writing chunk 2-0
Writing chunk 2-1
Writing chunk 2-2
Writing chunk 2-3
Writing chunk 3-0
Writing chunk 3-1
Writing chunk 3-2
Skipping chunk 3-3
